In [7]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import UnitaryGate

# =============================================================================
# 1. DEFINE YOUR INPUTS HERE
# =============================================================================
num_controls = 7 #This is the first input (n).

U_matrix = np.array([[0, 1], 
                     [1, 0]], dtype=complex) #This is the second input, the unitary matrix (U). It is set as the Pauli X-Matrix as a test.


# =============================================================================
# 2. THE ALGORITHM FUNCTION
# =============================================================================
def create_multi_controlled_u(n: int, matrix: np.ndarray) -> QuantumCircuit:
    if matrix.shape != (2, 2):
        raise ValueError("The matrix U must be of size 2x2.") #Check the matrix has the correct dimension. A single-qubit gate must be 2x2.
        
    identity_check = np.dot(matrix, matrix.conj().T)
    if not np.allclose(identity_check, np.eye(2)):
        raise ValueError("The provided matrix is NOT unitary.") #Check that the input matrix is indeed unitary.

    u_gate = UnitaryGate(matrix, label="U")
    ctrl_u_gate = u_gate.control(num_ctrl_qubits=1)
    
# Base Case 1: Zero control qubits (Direct application of U)
    if n == 0:
        qr_target = QuantumRegister(1, 'target')
        qc = QuantumCircuit(qr_target)
        qc.append(u_gate, [qr_target[0]])
        return qc
        
# Base Case 2: Exactly one control qubit (Direct application of C^1 U)
    if n == 1:
        qr_ctrl = QuantumRegister(1, 'control')
        qr_target = QuantumRegister(1, 'target')
        qc = QuantumCircuit(qr_ctrl, qr_target)
        qc.append(ctrl_u_gate, [qr_ctrl[0], qr_target[0]])
        return qc
        
# General Case: n >= 2
    qr_ctrl = QuantumRegister(n, 'control')
    qr_target = QuantumRegister(1, 'target')
    qr_anc = QuantumRegister(n - 1, 'ancilla')
    
    qc = QuantumCircuit(qr_ctrl, qr_target, qr_anc)

    qc.ccx(qr_ctrl[0], qr_ctrl[1], qr_anc[0])
    
    for i in range(2, n):
        qc.ccx(qr_anc[i-2], qr_ctrl[i], qr_anc[i-1])

    qc.append(ctrl_u_gate, [qr_anc[n-2], qr_target[0]])

    for i in range(n - 1, 1, -1):
        qc.ccx(qr_anc[i-2], qr_ctrl[i], qr_anc[i-1])
        
    qc.ccx(qr_ctrl[0], qr_ctrl[1], qr_anc[0])

    return qc


# =============================================================================
# 3. RUN AND SHOW RESULTS
# =============================================================================
# This runs automatically as soon as Python reaches this line
generated_circuit = create_multi_controlled_u(num_controls, U_matrix)

print(f"--- Multi-Controlled U Circuit Generated Successfully (n={num_controls}) ---")
print(f"Total Qubits: {generated_circuit.num_qubits} ({num_controls} Controls, 1 Target, {num_controls-1} Ancillas)")
print(f"Circuit Depth: {generated_circuit.depth()}")
print(f"Gate Counts: {dict(generated_circuit.count_ops())}")
print("\nText Representation of the Circuit:\n")
print(generated_circuit.draw(output='text'))

--- Multi-Controlled U Circuit Generated Successfully (n=7) ---
Total Qubits: 14 (7 Controls, 1 Target, 6 Ancillas)
Circuit Depth: 13
Gate Counts: {'ccx': 12, 'c-unitary': 1}

Text Representation of the Circuit:

                                                                            
control_0: ──■───────────────────────────────────────────────────────────■──
             │                                                           │  
control_1: ──■───────────────────────────────────────────────────────────■──
             │                                                           │  
control_2: ──┼────■─────────────────────────────────────────────────■────┼──
             │    │                                                 │    │  
control_3: ──┼────┼────■───────────────────────────────────────■────┼────┼──
             │    │    │                                       │    │    │  
control_4: ──┼────┼────┼────■─────────────────────────────■────┼────┼────┼──
             │   